# Exploratory Data Analysis

## Orchestra Subscription Prediction

The objective of this project is to predict whether previous orchestra patrons will purchase a subscription for the 2014–15 concert season.

This notebook explores the eight provided datasets to understand their structure, identify potential data quality issues, and prepare for feature engineering.

### Initial EDA
1. Load and inspect datasets
2. Examine dimensions, columns, and data types
3. Identify duplicate records
4. Analyze missing values

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

# Locate the raw data directory
DATA_DIR = Path("../data/raw")

if not DATA_DIR.exists():
    DATA_DIR = Path("data/raw")

assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

## 1. Load Datasets

Load the eight CSV files into pandas DataFrames.

The datasets contain:
- **train:** Account IDs and subscription purchase labels
- **test:** Account IDs requiring predictions
- **account:** Patron information, location, and donations
- **tickets_all:** Historical individual ticket purchases
- **subscriptions:** Historical season subscriptions
- **concerts:** Concert information from previous seasons
- **concerts_2014-15:** Planned concerts for the target season
- **zipcodes:** Geographic and demographic information

The sample submission file is excluded from EDA.

In [3]:
files = {
    "train": "train.csv",
    "test": "test.csv",
    "account": "account.csv",
    "tickets_all": "tickets_all.csv",
    "subscriptions": "subscriptions.csv",
    "concerts": "concerts.csv",
    "concerts_2014-15": "concerts_2014-15.csv",
    "zipcodes": "zipcodes.csv",
}

datasets = {}

for name, filename in files.items():
    path = DATA_DIR / filename

    # Preserve ZIP codes as strings to retain leading zeros
    dtype = {}
    if name == "account":
        dtype = {
            "shipping.zip.code": "string",
            "billing.zip.code": "string",
        }
    elif name == "zipcodes":
        dtype = {"Zipcode": "string"}

    try:
        df = pd.read_csv(path, dtype=dtype, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(path, dtype=dtype, encoding="cp1252")

    datasets[name] = df
    print(f"Loaded {filename}: {df.shape[0]:,} rows, {df.shape[1]} columns")

Loaded train.csv: 6,941 rows, 2 columns
Loaded test.csv: 2,975 rows, 1 columns
Loaded account.csv: 19,833 rows, 10 columns
Loaded tickets_all.csv: 2,808 rows, 8 columns
Loaded subscriptions.csv: 28,627 rows, 9 columns
Loaded concerts.csv: 103 rows, 6 columns
Loaded concerts_2014-15.csv: 6 rows, 5 columns
Loaded zipcodes.csv: 42,522 rows, 12 columns


## 2. Dataset Structure

First, examine the dimensions and structure of each dataset.

For each dataset, we will inspect:
- Number of rows and columns
- Column names and data types
- A sample of the first five observations

This helps establish what information is available and the level of detail represented by each row.

In [4]:
overview = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Memory (MB)": df.memory_usage(deep=True).sum() / 1024**2,
    }
    for name, df in datasets.items()
])

display(overview.round(2))

,Dataset,Rows,Columns,Memory (MB)
0,train,6941,2,0.53
1,test,2975,1,0.20
2,account,19833,10,7.31
3,tickets_all,2808,8,1.02
4,subscriptions,28627,9,11.33
5,concerts,103,6,0.07
6,concerts_2014-15,6,5,0.00
7,zipcodes,42522,12,17.46


In [5]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"DATASET: {name.upper()}")
    print(f"{'=' * 60}")

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

    print("\nColumn types:")
    display(pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
    }))

    print("\nFirst 5 rows:")
    display(df.head())


DATASET: TRAIN
Shape: 6,941 rows × 2 columns

Column types:


,Column,Data Type
0,account.id,object
1,label,int64



First 5 rows:


,account.id,label
0,001i000000NuRo3,0
1,001i000000NuRxd,0
2,001i000000NuQGN,0
3,001i000000NuPfL,0
4,001i000000NuQkP,0



DATASET: TEST
Shape: 2,975 rows × 1 columns

Column types:


,Column,Data Type
0,ID,object


... (22 outputs left out)


DATASET: ZIPCODES
Shape: 42,522 rows × 12 columns

Column types:


,Column,Data Type
0,Zipcode,string
1,ZipCodeType,object
2,City,object
3,State,object
4,LocationType,object
5,Lat,float64
6,Long,float64
7,Location,object
8,Decommisioned,bool
9,TaxReturnsFiled,float64



First 5 rows:


,Zipcode,ZipCodeType,City,State,LocationType,Lat,Long,Location,Decommisioned,TaxReturnsFiled,EstimatedPopulation,TotalWages
0,705,STANDARD,AIBONITO,PR,PRIMARY,18.14,-66.26,NA-US-PR-AIBONITO,False,NaN,NaN,NaN
1,610,STANDARD,ANASCO,PR,PRIMARY,18.28,-67.14,NA-US-PR-ANASCO,False,NaN,NaN,NaN
2,611,PO BOX,ANGELES,PR,PRIMARY,18.28,-66.79,NA-US-PR-ANGELES,False,NaN,NaN,NaN
3,612,STANDARD,ARECIBO,PR,PRIMARY,18.45,-66.73,NA-US-PR-ARECIBO,False,NaN,NaN,NaN
4,601,STANDARD,ADJUNTAS,PR,PRIMARY,18.16,-66.72,NA-US-PR-ADJUNTAS,False,NaN,NaN,NaN


## 3. Duplicate Records

Duplicate records can lead to incorrect statistics or double-counting during feature engineering.

We will check for:
1. **Exact duplicates:** Rows where every column has identical values.
2. **Duplicate identifiers:** Repeated account IDs or ZIP codes in tables where each identifier should be unique.

Importantly, repeated account IDs are expected in transaction-level datasets such as tickets and subscriptions because a patron may have multiple purchases.

In [6]:
duplicate_summary = []

for name, df in datasets.items():
    n_duplicates = df.duplicated().sum()

    duplicate_summary.append({
        "Dataset": name,
        "Total Rows": len(df),
        "Duplicate Rows": n_duplicates,
        "Duplicate (%)": 100 * n_duplicates / len(df),
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

display(duplicate_summary.round(2))

,Dataset,Total Rows,Duplicate Rows,Duplicate (%)
0,train,6941,0,0.0
1,test,2975,0,0.0
2,account,19833,0,0.0
3,tickets_all,2808,0,0.0
4,subscriptions,28627,0,0.0
5,concerts,103,0,0.0
6,concerts_2014-15,6,0,0.0
7,zipcodes,42522,0,0.0


In [7]:
# These datasets should have one row per identifier
unique_keys = {
    "train": "account.id",
    "test": "ID",
    "account": "account.id",
    "zipcodes": "Zipcode",
}

key_summary = []

for name, key in unique_keys.items():
    df = datasets[name]

    key_summary.append({
        "Dataset": name,
        "Key Column": key,
        "Total Rows": len(df),
        "Unique IDs": df[key].nunique(),
        "Duplicate IDs": df.duplicated(subset=[key]).sum(),
        "Missing IDs": df[key].isna().sum(),
    })

display(pd.DataFrame(key_summary))

,Dataset,Key Column,Total Rows,Unique IDs,Duplicate IDs,Missing IDs
0,train,account.id,6941,6941,0,0
1,test,ID,2975,2975,0,0
2,account,account.id,19833,19833,0,0
3,zipcodes,Zipcode,42522,42522,0,0


## 4. Missing Value Analysis

Missing values may affect feature engineering and model performance.

For each dataset, we will calculate:
- The number of missing values in each column
- The percentage of observations missing that value

This will help us determine which features are sufficiently complete and which may require further investigation or imputation.

At this stage, we will only identify missingness. No data will be modified or imputed.

In [8]:
missing_summary = []

for name, df in datasets.items():
    missing_cells = df.isna().sum().sum()
    total_cells = df.size

    missing_summary.append({
        "Dataset": name,
        "Total Cells": total_cells,
        "Missing Cells": missing_cells,
        "Missing (%)": 100 * missing_cells / total_cells,
        "Columns With Missing": df.isna().any().sum(),
    })

missing_summary = pd.DataFrame(missing_summary)

display(missing_summary.round(2))

,Dataset,Total Cells,Missing Cells,Missing (%),Columns With Missing
0,train,13882,0,0.00,0
1,test,2975,0,0.00,0
2,account,198330,77773,39.21,6
3,tickets_all,22464,2498,11.12,4
4,subscriptions,257643,8089,3.14,4
5,concerts,618,3,0.49,1
6,concerts_2014-15,30,0,0.00,0
7,zipcodes,510264,42261,8.28,6


In [9]:
for name, df in datasets.items():

    missing = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing (%)": df.isna().mean() * 100,
    })

    # Only display columns containing missing values
    missing = missing[missing["Missing Count"] > 0]

    missing = missing.sort_values(
        by="Missing (%)",
        ascending=False
    )

    print(f"\n{'=' * 60}")
    print(f"MISSING VALUES: {name.upper()}")
    print(f"{'=' * 60}")

    if missing.empty:
        print("No missing values found.")
    else:
        display(missing.round(2))


MISSING VALUES: TRAIN
No missing values found.

MISSING VALUES: TEST
No missing values found.

MISSING VALUES: ACCOUNT


,Missing Count,Missing (%)
shipping.zip.code,19569,98.67
shipping.city,19561,98.63
relationship,19172,96.67
first.donated,14298,72.09
billing.zip.code,2955,14.90
billing.city,2218,11.18



MISSING VALUES: TICKETS_ALL


,Missing Count,Missing (%)
marketing.source,2224,79.20
price.level,208,7.41
location,44,1.57
set,22,0.78



MISSING VALUES: SUBSCRIPTIONS


,Missing Count,Missing (%)
section,4543,15.87
price.level,3534,12.34
location,6,0.02
package,6,0.02



MISSING VALUES: CONCERTS


,Missing Count,Missing (%)
location,3,2.91



MISSING VALUES: CONCERTS_2014-15
No missing values found.

MISSING VALUES: ZIPCODES


,Missing Count,Missing (%)
TotalWages,13678,32.17
TaxReturnsFiled,13643,32.08
EstimatedPopulation,13643,32.08
Lat,648,1.52
Long,648,1.52
Location,1,0.00


## 5. Initial EDA Conclusions

- **Dataset Structure:** All eight datasets loaded successfully. The training set contains 6,941 labeled accounts, while the test set contains 2,975 accounts. Historical ticket and subscription records provide additional information for feature engineering.

- **Duplicates:** No exact duplicate rows were found in any dataset. All identifiers expected to be unique are indeed unique, with no missing IDs. Repeated account IDs in purchase-history tables are expected.

- **Missing Values:** Missingness varies significantly across datasets:
  - **Account:** 39.21% missing overall. Shipping information (~99%), relationship (96.67%), and first donation date (72.09%) are particularly sparse.
  - **Tickets:** Marketing source is 79.20% missing.
  - **Subscriptions:** Section (15.87%) and price level (12.34%) have moderate missingness.
  - **Zipcodes:** Demographic features have approximately 32% missingness.
  - **Train, test, and planned concerts:** No missing values.

- **Next Steps:** Investigate the meaning of missing values before deciding how to handle them. Explore relationships between datasets and historical purchasing behavior to identify useful predictive features. Ensure all features are available before the 2014–15 subscription purchasing period to prevent data leakage.

## 6. Target Variable Analysis

Examine the distribution of the target variable (`label`) to determine the proportion of customers who purchased a 2014–15 subscription.

This is important for understanding class imbalance and choosing appropriate modeling and validation strategies.

In [11]:
train = datasets["train"]

target_counts = train["label"].value_counts().sort_index()

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": (target_counts / len(train) * 100).round(2)
})

target_summary.index = ["No Subscription (0)", "Subscription (1)"]

display(target_summary)

,Count,Percentage
No Subscription (0),6609,95.22
Subscription (1),332,4.78


## 7. Feature Missingness Threshold

For initial feature screening, we will consider dropping columns with more than 75% missing values.

This threshold is a starting point and can be adjusted later. For now, we will identify these columns without modifying the original datasets.

In [12]:
MISSING_THRESHOLD = 0.75

high_missing_features = []

for name, df in datasets.items():
    missing_rates = df.isna().mean()

    for column, rate in missing_rates.items():
        if rate > MISSING_THRESHOLD:
            high_missing_features.append({
                "Dataset": name,
                "Feature": column,
                "Missing (%)": round(rate * 100, 2)
            })

high_missing_df = pd.DataFrame(high_missing_features)

if high_missing_df.empty:
    print("No features exceed the missingness threshold.")
else:
    display(
        high_missing_df.sort_values(
            "Missing (%)",
            ascending=False
        )
    )

,Dataset,Feature,Missing (%)
0,account,shipping.zip.code,98.67
1,account,shipping.city,98.63
2,account,relationship,96.67
3,tickets_all,marketing.source,79.20


## 8. Dataset Coverage

Determine how much historical information is available for the accounts in the training and test sets.

Specifically, examine what percentage of accounts have:
- Account information
- Previous subscription records
- Previous individual ticket purchases

Comparing train and test coverage also helps identify potential differences between the two populations.

In [13]:
source_ids = {
    "account": set(datasets["account"]["account.id"].dropna().astype(str)),
    "subscriptions": set(datasets["subscriptions"]["account.id"].dropna().astype(str)),
    "tickets_all": set(datasets["tickets_all"]["account.id"].dropna().astype(str)),
}

coverage_rows = []

for split, id_column in [("train", "account.id"), ("test", "ID")]:
    ids = datasets[split][id_column].astype(str)

    row = {
        "Dataset": split,
        "Total Accounts": len(ids),
    }

    for source, source_id_set in source_ids.items():
        row[f"{source} Coverage (%)"] = round(
            ids.isin(source_id_set).mean() * 100, 2
        )

    coverage_rows.append(row)

display(pd.DataFrame(coverage_rows))

,Dataset,Total Accounts,account Coverage (%),subscriptions Coverage (%),tickets_all Coverage (%)
0,train,6941,100.0,31.77,10.47
1,test,2975,100.0,31.70,9.82


## 9. Historical Season Coverage and Leakage Check

Examine which concert seasons are represented in the historical datasets.

Since the target is purchasing a subscription for 2014–15, historical purchasing features should only use information available before that season's subscription decision.

We will inspect the season distributions and flag any historical records from 2014–15 or later. The planned 2014–15 concerts are treated separately.

In [14]:
historical_tables = {
    "subscriptions": datasets["subscriptions"],
    "tickets_all": datasets["tickets_all"],
    "concerts": datasets["concerts"],
}

season_counts = pd.DataFrame({
    name: df["season"].value_counts()
    for name, df in historical_tables.items()
}).fillna(0).astype(int).sort_index()

display(season_counts)

print("\nPotential future-season records:")

for name, df in historical_tables.items():
    season_year = pd.to_numeric(
        df["season"].astype(str).str[:4],
        errors="coerce"
    )

    future_count = (season_year >= 2014).sum()

    print(f"{name}: {future_count:,} records from 2014–15 or later")

,subscriptions,tickets_all,concerts
season,,,
1993-1994,1624,0,0
1994-1995,1842,0,0
1995-1996,1774,0,0
1996-1997,1871,0,0
1997-1998,1788,0,0
1998-1999,1943,0,0
1999-2000,1607,0,0
2000-2001,1704,0,0
2001-2002,1457,0,0



Potential future-season records:
subscriptions: 0 records from 2014–15 or later
tickets_all: 0 records from 2014–15 or later
concerts: 0 records from 2014–15 or later


## 10. Historical Purchasing Behavior

Investigate whether previous purchasing activity is associated with subscription purchases in 2014–15.

We will compare subscription purchase rates across customers with:
- Previous subscriptions
- Previous individual ticket purchases
- Both types of purchase history
- No recorded purchase history

These relationships can help identify promising features for our model.

In [15]:
# Aggregate historical subscriptions by account
subs = datasets["subscriptions"].copy()

subs["season_start"] = pd.to_numeric(
    subs["season"].astype(str).str[:4],
    errors="coerce"
)

subscription_history = subs.groupby("account.id").agg(
    prior_subscriptions=("season", "size"),
    subscription_seasons=("season", "nunique"),
    last_subscription_year=("season_start", "max")
)

# Join historical summaries to labeled accounts
train_hist = train.merge(
    subscription_history,
    left_on="account.id",
    right_index=True,
    how="left",
    validate="one_to_one"
)

train_hist["has_prior_subscription"] = (
    train_hist["prior_subscriptions"].notna()
)

ticket_ids = set(
    datasets["tickets_all"]["account.id"].dropna().astype(str)
)

train_hist["has_prior_ticket"] = (
    train_hist["account.id"].astype(str).isin(ticket_ids)
)

# Helper for comparing purchase rates across groups
def purchase_rate_table(df, group_columns):
    result = df.groupby(
        group_columns, observed=True
    )["label"].agg(
        accounts="size",
        buyers="sum",
        rate="mean"
    )

    result["Purchase Rate (%)"] = (
        result.pop("rate") * 100
    ).round(2)

    return result

display(
    purchase_rate_table(
        train_hist,
        ["has_prior_subscription", "has_prior_ticket"]
    )
)

accounts  buyers  Purchase Rate (%)
has_prior_subscription has_prior_ticket                                     
False                  False                 4158      20               0.48
                       True                   578      12               2.08
True                   False                 2056     257              12.50
                       True                   149      43              28.86

## 11. Subscription Recency and Frequency

For customers with previous subscriptions, examine whether their likelihood of purchasing again depends on:

1. **Recency:** How many seasons have passed since their most recent subscription?
2. **Frequency:** How many distinct seasons have they previously subscribed to?

These variables may capture customer loyalty, retention, and disengagement.

In [16]:
# Recency relative to the target 2014–15 season
train_hist["years_since_last_sub"] = (
    2014 - train_hist["last_subscription_year"]
)

train_hist["recency_group"] = pd.cut(
    train_hist["years_since_last_sub"],
    bins=[0, 1, 2, 3, 5, 10, float("inf")],
    labels=["1", "2", "3", "4–5", "6–10", "11+"]
)

train_hist["recency_group"] = (
    train_hist["recency_group"]
    .cat.add_categories("No prior subscription")
    .fillna("No prior subscription")
)

print("Purchase rate by subscription recency:")
display(purchase_rate_table(train_hist, "recency_group"))


# Number of distinct subscription seasons
train_hist["subscription_seasons"] = (
    train_hist["subscription_seasons"].fillna(0)
)

train_hist["frequency_group"] = pd.cut(
    train_hist["subscription_seasons"],
    bins=[-0.1, 0, 1, 2, 5, float("inf")],
    labels=["0", "1", "2", "3–5", "6+"]
)

print("\nPurchase rate by historical subscription frequency:")
display(purchase_rate_table(train_hist, "frequency_group"))

Purchase rate by subscription recency:


,accounts,buyers,Purchase Rate (%)
recency_group,,,
1,345,282,81.74
2,83,11,13.25
3,52,4,7.69
4–5,105,3,2.86
6–10,223,0,0.00
11+,1397,0,0.00
No prior subscription,4736,32,0.68



Purchase rate by historical subscription frequency:


,accounts,buyers,Purchase Rate (%)
frequency_group,,,
0,4736,32,0.68
1,897,27,3.01
2,338,23,6.80
3–5,431,39,9.05
6+,539,211,39.15


## 12. Donation History

Explore whether customers who have donated to the orchestra are more likely to purchase subscriptions.

Compare:
- Donations made in 2013
- Lifetime donation amounts
- Number of lifetime donations

We will examine these features by target label and compare subscription rates between customers who did and did not donate in 2013.

Before using lifetime totals as predictive features, we will need to confirm they do not include activity occurring after the prediction date.

In [17]:
donation_cols = [
    "amount.donated.2013",
    "amount.donated.lifetime",
    "no.donations.lifetime"
]

train_donations = train.merge(
    datasets["account"][["account.id"] + donation_cols],
    on="account.id",
    how="left",
    validate="one_to_one"
)

# Compare donation statistics by target
donation_stats = (
    train_donations
    .groupby("label")[donation_cols]
    .agg(["median", "mean"])
    .round(2)
)

print("Donation statistics by target label:")
display(donation_stats)


# Compare customers who donated in 2013
train_donations["donated_2013"] = (
    train_donations["amount.donated.2013"].fillna(0) > 0
)

print("\nSubscription rate by 2013 donation activity:")
display(
    purchase_rate_table(
        train_donations,
        "donated_2013"
    )
)

Donation statistics by target label:


amount.donated.2013         amount.donated.lifetime           \
                   median    mean                  median     mean   
label                                                                
0                     0.0   26.04                     0.0   887.44   
1                    27.5  707.13                   887.5  8177.37   

      no.donations.lifetime         
                     median   mean  
label                               
0                       0.0   1.05  
1                      10.0  12.07


Subscription rate by 2013 donation activity:


,accounts,buyers,Purchase Rate (%)
donated_2013,,,
False,6682,160,2.39
True,259,172,66.41


## 13. Extended EDA Conclusions

The EDA reveals several strong relationships between historical customer behavior and subscription purchases.

- **Class Imbalance:** Only 332 of 6,941 training accounts (4.78%) purchased a 2014–15 subscription. This reinforces the importance of stratified validation and AUROC rather than accuracy.

- **Subscription Recency:** Recency is an exceptionally strong predictor. Customers who subscribed in 2013–14 renewed at an 81.74% rate, compared with 13.25% for those whose last subscription was two seasons earlier. No customers whose most recent subscription was six or more seasons ago purchased in the training set.

- **Subscription Frequency:** Customers with six or more previous subscription seasons had a 39.15% purchase rate, compared with just 0.68% for customers with no subscription history. Frequency and recency are likely correlated.

- **Other Engagement Signals:** Customers with both subscription and individual ticket histories purchased at a 28.86% rate, compared with 0.48% for those with neither. Customers who donated in 2013 purchased at a 66.41% rate, compared with 2.39% for non-donors. These relationships may overlap with recent subscription activity.

- **Data Quality:** Train and test have nearly identical historical data coverage, suggesting similar populations. Four features exceed our 75% missingness threshold. No records from 2014–15 or later were found in the historical purchase tables.

**Conclusion:** Recent subscription activity, long-term customer loyalty, and donation behavior appear to be our strongest predictive signals. The sharp decline in renewal rates suggests that distinguishing active subscribers from lapsed customers will be particularly valuable. These findings provide a strong foundation for account-level feature engineering.